# Structured streaming Demo

In [8]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType
from pyspark.sql.functions import avg

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [2]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [9]:
# Analyze the average change in page size for bot and non-bot edits
# size_change = new page size - old page size
# A positive value means that content was added, while a negative value means that content was removed
edit_size_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .withColumn(
        "size_change",
        col("length.new") - col("length.old")
    ) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edit_count"),
        avg("size_change").alias("avg_size_change")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count"),
        col("avg_size_change")
    )

In [10]:
# Start the streaming query to display the average page size change
# The results will appear in the Docker logs
query_edit_size = edit_size_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [15]:
# Change the aggregation window from 1 hour to 30 minutes
# This provides a more detailed view of editing activity over time
windowed_counts_30min = parsed_df \ #This is tumbling window
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "30 minutes"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [11]:
spark.streams.active

In [17]:
for query in spark.streams.active:
    query.stop()

In [18]:
spark.streams.active

[]

In [16]:
# Start the streaming query and display the aggregated results in the Docker logs
query_30min = windowed_counts_30min \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [20]:
# Create an overlapping (sliding) window of 30 minutes with a 10-minute slide.
# Each window covers 30 minutes of events and a new window starts every 10 minutes.
# Therefore, consecutive windows overlap.

overlapping_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(
            col("event_time"),
            "30 minutes",
            "10 minutes"
        ),
        col("bot")
    ) \
    .agg(
        count("*").alias("edit_count")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [21]:
query_overlapping = overlapping_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()